In [1]:
# 1. INSTALACIÓN DE MÓDULOS (Limpia y rápida)

%pip install -q amplpy ipywidgets

!python -m amplpy.modules install cplex



import pandas as pd

from amplpy import AMPL

from IPython.display import display, HTML

import ipywidgets as widgets



# 2. INICIALIZACIÓN Y CONFIGURACIÓN DE PANDAS

ampl = AMPL()

pd.options.display.float_format = '{:,.2f}'.format # Números reales con comas

pd.set_option('display.max_rows', None)



# 3. DEFINICIÓN DEL MODELO AMPL (.mod)

ampl.eval(r"""

set I; set J; set T ordered; set K;

param d {i in I, t in T} >= 0;

param r {i in I, j in J} > 0;

param mu {j in J} >= 0;

param Tdisp >= 0;

param B {j in J} >= 0, integer;

param C0 {j in J} >= 0, integer;

param Obar {j in J} >= 0;

param CI {j in J, t in T} >= 0;

param Cmo {t in T} >= 0 default 0;

param Cp {i in I, j in J, t in T} >= 0 default 0;

param Cop {j in J, t in T}  >= 0 default 0;

param Ch                       >= 0 default 0;

param Cf                       >= 0 default 0;

param Nmax {j in J} >= 0, integer;



var X  {i in I, j in J, t in T} >= 0;

var ND {j in J, t in T}         >= 0, integer;

var NN {j in J, t in T}         >= 0, integer;

var O  {j in J, t in T}         >= 0, integer;

var H  {j in J, t in T}         >= 0, integer;

var L  {j in J, t in T}         >= 0, integer;

var y  {j in J, t in T}         >= 0, integer;

var Wturns {t in T} >= 0, integer;

var w {k in K, t in T} binary;

var ZNN {j in J, t in T, k in K} >= 0;



minimize TotalCost:

    sum {j in J, t in T} y[j,t] * CI[j,t]

  + sum {j in J, t in T} (ND[j,t] - NN[j,t]) * Cop[j,t]

  + sum {j in J, t in T} O[j,t] * Cmo[t]

  + sum {j in J, t in T} (H[j,t] * Ch + L[j,t] * Cf)

  + sum {i in I, j in J, t in T} X[i,j,t] * Cp[i,j,t];



s.t. OneShiftPerPeriod {t in T}: sum {k in K} w[k,t] = 1;

s.t. Demand {i in I, t in T}: sum {j in J} X[i,j,t] >= d[i,t];

s.t. Capacity {j in J, t in T}:

    sum {i in I} X[i,j,t] / r[i,j] <= Tdisp * mu[j] * (sum {k in K} k * ZNN[j,t,k]);

s.t. Install_1 {j in J, t in T: ord(t)=1}: ND[j,t] = B[j] + y[j,t];

s.t. Install_t {j in J, t in T: ord(t)>1}: ND[j,t] = ND[j, prev(t)] + y[j,t];

s.t. Active_le_install {j in J, t in T}: NN[j,t] <= ND[j,t];

s.t. Active_le_Nmax {j in J, t in T}: NN[j,t] <= Nmax[j];

s.t. Labor_1 {j in J, t in T: ord(t)=1}: O[j,t] = C0[j] + H[j,t] - L[j,t];

s.t. Labor_t {j in J, t in T: ord(t)>1}: O[j,t] = O[j, prev(t)] + H[j,t] - L[j,t];

s.t. Labor_req {j in J, t in T}: O[j,t] = Obar[j] * (sum {k in K} k * ZNN[j,t,k]);

s.t. ZNN_up_by_w {j in J, t in T, k in K}: ZNN[j,t,k] <= Nmax[j] * w[k,t];

s.t. ZNN_up_by_NN {j in J, t in T, k in K}: ZNN[j,t,k] <= NN[j,t];

s.t. ZNN_lo_by_wNN {j in J, t in T, k in K}: ZNN[j,t,k] >= NN[j,t] - (1 - w[k,t]) * Nmax[j];

s.t. Wturns_def {t in T}: Wturns[t] = sum {k in K} k * w[k,t];

""")



# 4. CARGA DE DATOS (.dat)

with open('data.dat', 'w') as f:

    f.write(r"""

set I := 1 2 3 4;

set J := 1 2 3;

set T := 1 2 3 4 5 6 7 8 9 10;

set K := 1 2 3;

param Tdisp := 2080;

param mu := 1 0.80 2 0.90 3 0.98;

param Obar := 1 3 2 2 3 1;

param C0 := 1 3 2 0 3 0;

param B := 1 1 2 0 3 0;

param d : 1 2 3 4 5 6 7 8 9 10 :=

1 100000 111000 122000 138000 177000 236000 330000 411000 483000 605000

2 328000 420000 549000 662000 788000 950000 1091000 1479000 1651000 1830000

3 367000 470000 650000 762000 1021000 1140000 1293000 1736000 2227000 3111000

4 180000 226000 308000 391000  523000 686000 942000 1089000 1452000 1815000;

param r : 1 2 3 := 1 120 120 120 2 170 170 170 3 400 400 400 4 600 600 600;

param CI : 1 2 3 4 5 6 7 8 9 10 :=

1 25000.00 21739.13 18903.59 16437.91 14293.83 12429.42 10808.19 9398.43 8172.54 7106.56

2 50000.00 43478.26 37807.18 32875.81 28587.66 24858.84 21616.38 18796.85 16345.09 14213.12

3 75000.00 65217.39 56710.78 49313.72 42881.49 37288.26 32424.57 28195.28 24517.63 21319.68;

param Cmo := 1 16032.00 2 13940.87 3 12122.50 4 10541.30 5 9166.35 6 7970.74 7 6931.08 8 6027.02 9 5240.89 10 4557.29;

param Cop : 1 2 3 4 5 6 7 8 9 10 :=

1 2500.00 2173.91 1890.36 1643.79 1429.38 1242.94 1080.82 939.84 817.25 710.66

2 5000.00 4347.83 3780.72 3287.58 2858.77 2485.88 2161.64 1879.69 1634.51 1421.31

3 7500.00 6521.74 5671.08 4931.37 4288.15 3728.83 3242.46 2819.53 2451.76 2131.97;

param Nmax := 1 1000 2 1000 3 1000;

param Ch := 500;

param Cf := 4500;

param Cp :=

[1,1,1] 1.00 [1,1,2] 0.87 [1,1,3] 0.76 [1,1,4] 0.66 [1,1,5] 0.57 [1,1,6] 0.50 [1,1,7] 0.43 [1,1,8] 0.38 [1,1,9] 0.33 [1,1,10] 0.28

[1,2,1] 0.75 [1,2,2] 0.65 [1,2,3] 0.57 [1,2,4] 0.49 [1,2,5] 0.43 [1,2,6] 0.37 [1,2,7] 0.32 [1,2,8] 0.28 [1,2,9] 0.25 [1,2,10] 0.21

[1,3,1] 0.50 [1,3,2] 0.43 [1,3,3] 0.38 [1,3,4] 0.33 [1,3,5] 0.29 [1,3,6] 0.25 [1,3,7] 0.22 [1,3,8] 0.19 [1,3,9] 0.16 [1,3,10] 0.14

[2,1,1] 1.00 [2,1,2] 0.87 [2,1,3] 0.76 [2,1,4] 0.66 [2,1,5] 0.57 [2,1,6] 0.50 [2,1,7] 0.43 [2,1,8] 0.38 [2,1,9] 0.33 [2,1,10] 0.28

[2,2,1] 0.75 [2,2,2] 0.65 [2,2,3] 0.57 [2,2,4] 0.49 [2,2,5] 0.43 [2,2,6] 0.37 [2,2,7] 0.32 [2,2,8] 0.28 [2,2,9] 0.25 [2,2,10] 0.21

[2,3,1] 0.50 [2,3,2] 0.43 [2,3,3] 0.38 [2,3,4] 0.33 [2,3,5] 0.29 [2,3,6] 0.25 [2,3,7] 0.22 [2,3,8] 0.19 [2,3,9] 0.16 [2,3,10] 0.14

[3,1,1] 1.00 [3,1,2] 0.87 [3,1,3] 0.76 [3,1,4] 0.66 [3,1,5] 0.57 [3,1,6] 0.50 [3,1,7] 0.43 [3,1,8] 0.38 [3,1,9] 0.33 [3,1,10] 0.28

[3,2,1] 0.75 [3,2,2] 0.65 [3,2,3] 0.57 [3,2,4] 0.49 [3,2,5] 0.43 [3,2,6] 0.37 [3,2,7] 0.32 [3,2,8] 0.28 [3,2,9] 0.25 [3,2,10] 0.21

[3,3,1] 0.50 [3,3,2] 0.43 [3,3,3] 0.38 [3,3,4] 0.33 [3,3,5] 0.29 [3,3,6] 0.25 [3,3,7] 0.22 [3,3,8] 0.19 [3,3,9] 0.16 [3,3,10] 0.14

[4,1,1] 1.00 [4,1,2] 0.87 [4,1,3] 0.76 [4,1,4] 0.66 [4,1,5] 0.57 [4,1,6] 0.50 [4,1,7] 0.43 [4,1,8] 0.38 [4,1,9] 0.33 [4,1,10] 0.28

[4,2,1] 0.75 [4,2,2] 0.65 [4,2,3] 0.57 [4,2,4] 0.49 [4,2,5] 0.43 [4,2,6] 0.37 [4,2,7] 0.32 [4,2,8] 0.28 [4,2,9] 0.25 [4,2,10] 0.21

[4,3,1] 0.50 [4,3,2] 0.43 [4,3,3] 0.38 [4,3,4] 0.33 [4,3,5] 0.29 [4,3,6] 0.25 [4,3,7] 0.22 [4,3,8] 0.19 [4,3,9] 0.16 [4,3,10] 0.14;

""")

ampl.read_data('data.dat')



# 5. EJECUCIÓN CON CPLEX

ampl.set_option('solver', 'cplex')

ampl.set_option('solver_msg', 1)

print("\n--- Ejecutando Solve ---")

ampl.solve()



# 6. IMPLEMENTACIÓN VISUAL (Pestañas y Estilos)

def color_zeros(val):

    color = '#D3D3D3' if val == 0 else 'black'

    return f'color: {color}'



variables = ['Wturns', 'X', 'ND', 'NN', 'H', 'L', 'y', 'O']

dfs_estilizados = {}



for var_name in variables:

    df = ampl.get_variable(var_name).get_values().to_pandas()

    # Estilo: Ceros grises y formato con comas

    styler = df.style.applymap(color_zeros).format("{:,.2f}")

    # Barras visuales para ver magnitud en variables clave

    if var_name in ['X', 'O', 'ND', 'NN']:

        styler = styler.bar(subset=[df.columns[0]], color='#AED6F1')

    dfs_estilizados[var_name] = styler



# Crear Tabs

tabs = []

for var_name in variables:

    out = widgets.Output()

    with out:

        display(HTML(f"<h3>Detalle de variable: {var_name}</h3>"))

        display(dfs_estilizados[var_name])

    tabs.append(out)



tab_control = widgets.Tab(children=tabs)

for i, var_name in enumerate(variables):

    tab_control.set_title(i, var_name)



# Mostrar Encabezado y Interfaz

header_html = f"""

<div style="background-color: #1B4F72; padding: 25px; border-radius: 12px; color: white; font-family: sans-serif;">

    <h1 style="margin: 0;">Reporte de Optimización Industrial</h1>

    <hr style="border: 0.5px solid #5DADE2;">

    <p style="font-size: 20px;">Costo Total Óptimo: <span style="color: #F7DC6F;"><b>${ampl.get_objective('TotalCost').value():,.2f}</b></span></p>

    <p style="font-size: 14px;">Solver utilizado: <b>CPLEX</b></p>

</div>

"""



display(HTML(header_html))

display(tab_control)

$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_cplex
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_cplex.

--- Ejecutando Solve ---
CPLEX 22.1.2CPLEX 22.1.2: optimal solution within tolerance; objective 8180658.89
18023 simplex iterations
3783 branching nodes
absmipgap=772.884, relmipgap=9.4477e-05


/tmp/ipykernel_1619/217337466.py:261: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(color_zeros).format("{:,.2f}")
